<a href="https://colab.research.google.com/github/daniyalshynbay/css314_15.10.2026/blob/main/CUDA_Python_50_Lab_Tasksheet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1: Environment Provisioning & GPU Verification
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv
!pip install -q numba cupy-cuda12x pytest pynvml

import numba.cuda as cuda
import cupy as cp

if not cuda.is_available():
    raise SystemError("CRITICAL: CUDA GPU not detected! Go to Runtime > Change runtime type > Select T4 GPU.")

gpu = cuda.get_current_device()

# Standardize device name handling (handles both str and bytes gracefully)
gpu_name = gpu.name.decode('utf-8') if isinstance(gpu.name, bytes) else gpu.name
cupy_gpu_name = cp.cuda.runtime.getDeviceProperties(0)['name']
cupy_gpu_name = cupy_gpu_name.decode('utf-8') if isinstance(cupy_gpu_name, bytes) else cupy_gpu_name

print(f"\n[+] Environment Certified for Autonomous CUDA Lab")
print(f"Target Device      : {gpu_name}")
print(f"Compute Capable    : {gpu.compute_capability}")
print(f"CuPy Device Name   : {cupy_gpu_name}")
print(f"Warp Size          : {gpu.WARP_SIZE} threads")
print(f"Max Block Threads  : {gpu.MAX_THREADS_PER_BLOCK}")
print(f"Max Shared/Block   : {gpu.MAX_SHARED_MEMORY_PER_BLOCK / 1024:.1f} KB\n")

name, driver_version, memory.total [MiB]
Tesla T4, 580.82.07, 15360 MiB

[+] Environment Certified for Autonomous CUDA Lab
Target Device      : Tesla T4
Compute Capable    : ComputeCapability(major=7, minor=5)
CuPy Device Name   : Tesla T4
Warp Size          : 32 threads
Max Block Threads  : 1024
Max Shared/Block   : 48.0 KB



In [ ]:
"""
CUDA Python Lab - Tasks 01 to 15 (Module 1 & Module 2)
Verified and tested for static shared memory shapes and exact numerical accuracy.
"""

import math
import numpy as np
from numba import cuda, float32, int64

# Verify CUDA runtime availability
if not cuda.is_available():
    raise SystemError("CRITICAL: CUDA GPU not detected! Enable GPU runtime before running.")


# ==============================================================================
# MODULE 1: CUDA EXECUTION MODEL (Tasks 01 - 08)
# ==============================================================================

# Task 01: 1D Vector Addition with Boundary Checks
@cuda.jit
def vector_add_1d(a, b, out):
    i = cuda.grid(1)
    if i < a.shape[0]:
        out[i] = a[i] + b[i]


# Task 02: 1D Grid-Stride Loop Element-wise Scaling
@cuda.jit
def vector_scale_grid_stride(x, alpha, out):
    start = cuda.grid(1)
    stride = cuda.gridsize(1)
    for i in range(start, x.shape[0], stride):
        out[i] = x[i] * alpha


# Task 03: 2D Matrix Element-wise Addition & Index Mapping
@cuda.jit
def matrix_add_2d(A, B, C):
    row, col = cuda.grid(2)
    if row < A.shape[0] and col < A.shape[1]:
        C[row, col] = A[row, col] + B[row, col]


# Task 04: 2D Matrix Transpose (Naive Global Memory)
@cuda.jit
def matrix_transpose_naive(A, B):
    row, col = cuda.grid(2)
    if row < A.shape[0] and col < A.shape[1]:
        B[col, row] = A[row, col]


# Task 05: 3D Grid Volume Initialization & Index Flattening
@cuda.jit
def volume_init_3d(vol, D, H, W):
    x, y, z = cuda.grid(3)
    if z < D and y < H and x < W:
        vol[z, y, x] = z * 10000 + y * 100 + x


# Task 06: Color Image to Grayscale Luminance Conversion
@cuda.jit
def rgb_to_grayscale(img_rgb, img_gray):
    y, x = cuda.grid(2)
    if y < img_rgb.shape[0] and x < img_rgb.shape[1]:
        r = img_rgb[y, x, 0]
        g = img_rgb[y, x, 1]
        b = img_rgb[y, x, 2]
        img_gray[y, x] = 0.299 * r + 0.587 * g + 0.114 * b


# Task 07: Vector Linear Interpolation (LERP) with Dynamic Step Size
@cuda.jit
def vector_lerp(a, b, t, out):
    i = cuda.grid(1)
    if i < a.shape[0]:
        out[i] = (1.0 - t[i]) * a[i] + t[i] * b[i]


# Task 08: Thresholding & Boolean Mask Generation
@cuda.jit
def vector_threshold(x, thresh, mask):
    start = cuda.grid(1)
    stride = cuda.gridsize(1)
    for i in range(start, x.shape[0], stride):
        if x[i] >= thresh:
            mask[i] = np.uint8(1)
        else:
            mask[i] = np.uint8(0)


# ==============================================================================
# MODULE 2: MEMORY HIERARCHY & SHARED MEMORY (Tasks 09 - 15)
# ==============================================================================

# Task 09: Coalesced vs Strided Memory Access Benchmark Kernel
@cuda.jit
def benchmark_strided_access(in_arr, out_arr, stride):
    i = cuda.grid(1)
    n = out_arr.shape[0]
    if i < n:
        src_idx = (i * stride) % in_arr.shape[0]
        out_arr[i] = in_arr[src_idx]


# Task 10: 1D Stencil / Moving Average with Constant Halo Bounds
@cuda.jit
def stencil_1d_shared(in_arr, out_arr, radius):
    # Static compile-time literal allocation for shared memory (256 + 2*16)
    s_data = cuda.shared.array(288, dtype=float32)

    tx = cuda.threadIdx.x
    i = cuda.grid(1)
    n = in_arr.shape[0]
    shared_idx = tx + radius

    if i < n:
        s_data[shared_idx] = in_arr[i]
    else:
        s_data[shared_idx] = in_arr[n - 1] if n > 0 else 0.0

    if tx < radius:
        left_idx = i - radius
        s_data[tx] = in_arr[max(0, left_idx)]

    if tx >= cuda.blockDim.x - radius:
        right_idx = i + radius
        s_data[shared_idx + radius] = in_arr[min(n - 1, right_idx)]

    cuda.syncthreads()

    if i < n:
        acc = 0.0
        for k in range(-radius, radius + 1):
            acc += s_data[shared_idx + k]
        out_arr[i] = acc / (2 * radius + 1)


# Task 11: 1D Convolution with Static Shared Memory Tiling
@cuda.jit
def conv_1d_shared_tiled(x, kernel, y):
    s_x = cuda.shared.array(320, dtype=float32)  # 256 + 64
    s_k = cuda.shared.array(64, dtype=float32)

    tx = cuda.threadIdx.x
    i = cuda.grid(1)
    N = x.shape[0]
    K = kernel.shape[0]
    half_k = K // 2

    if tx < K:
        s_k[tx] = kernel[tx]

    shared_idx = tx + half_k
    if i < N:
        s_x[shared_idx] = x[i]
    else:
        s_x[shared_idx] = 0.0

    if tx < half_k:
        left_i = i - half_k
        s_x[tx] = x[left_i] if left_i >= 0 else 0.0

    if tx >= cuda.blockDim.x - half_k:
        right_i = i + half_k
        s_x[shared_idx + half_k] = x[right_i] if right_i < N else 0.0

    cuda.syncthreads()

    if i < N:
        acc = 0.0
        for k in range(K):
            acc += s_x[tx + k] * s_k[K - 1 - k]
        y[i] = acc


# Task 12: 2D Tiled Matrix Multiplication (Square Tiles)
@cuda.jit
def matmul_tiled_shared(A, B, C):
    sA = cuda.shared.array((16, 16), dtype=float32)
    sB = cuda.shared.array((16, 16), dtype=float32)

    tx = cuda.threadIdx.x
    ty = cuda.threadIdx.y
    row = cuda.blockIdx.y * 16 + ty
    col = cuda.blockIdx.x * 16 + tx

    M, K_dim = A.shape
    _, N = B.shape
    acc = 0.0
    num_phased_tiles = (K_dim + 15) // 16

    for p in range(num_phased_tiles):
        if row < M and (p * 16 + tx) < K_dim:
            sA[ty, tx] = A[row, p * 16 + tx]
        else:
            sA[ty, tx] = 0.0

        if col < N and (p * 16 + ty) < K_dim:
            sB[ty, tx] = B[p * 16 + ty, col]
        else:
            sB[ty, tx] = 0.0

        cuda.syncthreads()

        for k in range(16):
            acc += sA[ty, k] * sB[k, tx]

        cuda.syncthreads()

    if row < M and col < N:
        C[row, col] = acc


# Task 13: 2D Bank Conflict-Free Matrix Transpose
@cuda.jit
def transpose_bank_conflict_free(A, B):
    # Padding (+1 column) eliminates 32-way shared memory bank conflicts
    tile = cuda.shared.array((32, 33), dtype=float32)
    tx = cuda.threadIdx.x
    ty = cuda.threadIdx.y

    x = cuda.blockIdx.x * 32 + tx
    y = cuda.blockIdx.y * 32 + ty
    M, N = A.shape

    if x < N and y < M:
        tile[ty, tx] = A[y, x]

    cuda.syncthreads()

    x_trans = cuda.blockIdx.y * 32 + tx
    y_trans = cuda.blockIdx.x * 32 + ty

    if x_trans < M and y_trans < N:
        B[y_trans, x_trans] = tile[tx, ty]


# Task 14: Parallel Tree Sum Reduction in Shared Memory
@cuda.jit
def reduction_sum_shared(in_arr, block_sums):
    s_data = cuda.shared.array(256, dtype=float32)
    tx = cuda.threadIdx.x
    i = cuda.grid(1)
    N = in_arr.shape[0]

    s_data[tx] = in_arr[i] if i < N else 0.0
    cuda.syncthreads()

    stride = cuda.blockDim.x // 2
    while stride > 0:
        if tx < stride:
            s_data[tx] += s_data[tx + stride]
        cuda.syncthreads()
        stride >>= 1

    if tx == 0:
        block_sums[cuda.blockIdx.x] = s_data[0]


# Task 15: Parallel Min/Max Reduction with Index Tracking
@cuda.jit
def reduction_argmax_shared(in_arr, block_max_vals, block_max_indices):
    s_vals = cuda.shared.array(256, dtype=float32)
    s_idxs = cuda.shared.array(256, dtype=int64)

    tx = cuda.threadIdx.x
    i = cuda.grid(1)
    N = in_arr.shape[0]

    if i < N:
        s_vals[tx] = in_arr[i]
        s_idxs[tx] = i
    else:
        s_vals[tx] = -1e38
        s_idxs[tx] = -1

    cuda.syncthreads()

    stride = cuda.blockDim.x // 2
    while stride > 0:
        if tx < stride:
            val_other = s_vals[tx + stride]
            idx_other = s_idxs[tx + stride]
            if val_other > s_vals[tx] or (val_other == s_vals[tx] and idx_other < s_idxs[tx]):
                s_vals[tx] = val_other
                s_idxs[tx] = idx_other
        cuda.syncthreads()
        stride >>= 1

    if tx == 0:
        block_max_vals[cuda.blockIdx.x] = s_vals[0]
        block_max_indices[cuda.blockIdx.x] = s_idxs[0]


# ==============================================================================
# INTEGRATED VERIFICATION SUITE
# ==============================================================================

if __name__ == "__main__":
    print("==================================================================")
    print("         RUNNING VERIFICATION FOR TASKS 1 TO 15                   ")
    print("==================================================================")

    # --- Task 01 ---
    N = 1000003
    a = np.random.randn(N).astype(np.float32)
    b = np.random.randn(N).astype(np.float32)
    out = np.zeros(N, dtype=np.float32)
    vector_add_1d[math.ceil(N / 256), 256](a, b, out)
    np.testing.assert_allclose(out, a + b, atol=1e-5)
    print("[PASSED] Task 01: vector_add_1d")

    # --- Task 02 ---
    N = 500000
    x = np.random.randn(N).astype(np.float32)
    out = np.zeros(N, dtype=np.float32)
    vector_scale_grid_stride[64, 256](x, 2.5, out)
    np.testing.assert_allclose(out, x * 2.5, atol=1e-5)
    print("[PASSED] Task 02: vector_scale_grid_stride")

    # --- Task 03 ---
    M, N = 513, 1021
    A = np.random.randn(M, N).astype(np.float32)
    B = np.random.randn(M, N).astype(np.float32)
    C = np.zeros((M, N), dtype=np.float32)
    matrix_add_2d[(math.ceil(M / 16), math.ceil(N / 16)), (16, 16)](A, B, C)
    np.testing.assert_allclose(C, A + B, atol=1e-5)
    print("[PASSED] Task 03: matrix_add_2d")

    # --- Task 04 ---
    M, N = 384, 512
    A = np.random.randn(M, N).astype(np.float32)
    B = np.zeros((N, M), dtype=np.float32)
    matrix_transpose_naive[(math.ceil(M / 16), math.ceil(N / 16)), (16, 16)](A, B)
    np.testing.assert_allclose(B, A.T, atol=1e-5)
    print("[PASSED] Task 04: matrix_transpose_naive")

    # --- Task 05 ---
    D, H, W = 33, 47, 65
    vol = np.zeros((D, H, W), dtype=np.float32)
    volume_init_3d[(math.ceil(W / 8), math.ceil(H / 8), math.ceil(D / 8)), (8, 8, 8)](vol, D, H, W)
    ref_vol = np.fromfunction(lambda z, y, x: z * 10000 + y * 100 + x, (D, H, W), dtype=np.float32)
    np.testing.assert_equal(vol, ref_vol)
    print("[PASSED] Task 05: volume_init_3d")

    # --- Task 06 ---
    H, W = 256, 256
    img_rgb = np.random.rand(H, W, 3).astype(np.float32)
    img_gray = np.zeros((H, W), dtype=np.float32)
    rgb_to_grayscale[(math.ceil(H / 16), math.ceil(W / 16)), (16, 16)](img_rgb, img_gray)
    ref_gray = 0.299 * img_rgb[:, :, 0] + 0.587 * img_rgb[:, :, 1] + 0.114 * img_rgb[:, :, 2]
    np.testing.assert_allclose(img_gray, ref_gray, atol=1e-4)
    print("[PASSED] Task 06: rgb_to_grayscale")

    # --- Task 07 ---
    N = 100000
    a = np.random.randn(N).astype(np.float32)
    b = np.random.randn(N).astype(np.float32)
    t = np.random.rand(N).astype(np.float32)
    out = np.zeros(N, dtype=np.float32)
    vector_lerp[math.ceil(N / 256), 256](a, b, t, out)
    np.testing.assert_allclose(out, (1.0 - t) * a + t * b, atol=1e-5)
    print("[PASSED] Task 07: vector_lerp")

    # --- Task 08 ---
    N = 100000
    x = np.random.randn(N).astype(np.float32)
    mask = np.zeros(N, dtype=np.uint8)
    vector_threshold[64, 256](x, 0.5, mask)
    np.testing.assert_equal(mask, (x >= 0.5).astype(np.uint8))
    print("[PASSED] Task 08: vector_threshold")

    # --- Task 09 ---
    N = 1024
    in_arr = np.arange(N, dtype=np.float32)
    out_arr = np.zeros(N, dtype=np.float32)
    stride = 4
    benchmark_strided_access[math.ceil(N / 256), 256](in_arr, out_arr, stride)
    ref_out = in_arr[(np.arange(N) * stride) % N]
    np.testing.assert_allclose(out_arr, ref_out, atol=1e-5)
    print("[PASSED] Task 09: benchmark_strided_access")

    # --- Task 10 ---
    N = 1024
    radius = 3
    in_arr = np.random.randn(N).astype(np.float32)
    out_arr = np.zeros(N, dtype=np.float32)
    stencil_1d_shared[math.ceil(N / 256), 256](in_arr, out_arr, radius)
    padded = np.pad(in_arr, (radius, radius), mode='edge')
    ref_out = np.convolve(padded, np.ones(2 * radius + 1) / (2 * radius + 1), mode='valid')
    np.testing.assert_allclose(out_arr, ref_out, atol=1e-4)
    print("[PASSED] Task 10: stencil_1d_shared")

    # --- Task 11 ---
    N = 1024
    K = 5
    x = np.random.randn(N).astype(np.float32)
    kernel = np.random.randn(K).astype(np.float32)
    y = np.zeros(N, dtype=np.float32)
    conv_1d_shared_tiled[math.ceil(N / 256), 256](x, kernel, y)
    ref_y = np.convolve(x, kernel, mode='same')
    np.testing.assert_allclose(y, ref_y, atol=1e-4)
    print("[PASSED] Task 11: conv_1d_shared_tiled")

    # --- Task 12 ---
    M, K_dim, N = 128, 256, 128
    A = np.random.randn(M, K_dim).astype(np.float32)
    B = np.random.randn(K_dim, N).astype(np.float32)
    C = np.zeros((M, N), dtype=np.float32)
    matmul_tiled_shared[(math.ceil(N / 16), math.ceil(M / 16)), (16, 16)](A, B, C)
    np.testing.assert_allclose(C, A @ B, atol=1e-3)
    print("[PASSED] Task 12: matmul_tiled_shared")

    # --- Task 13 ---
    M, N = 128, 256
    A = np.random.randn(M, N).astype(np.float32)
    B = np.zeros((N, M), dtype=np.float32)
    transpose_bank_conflict_free[(math.ceil(N / 32), math.ceil(M / 32)), (32, 32)](A, B)
    np.testing.assert_allclose(B, A.T, atol=1e-5)
    print("[PASSED] Task 13: transpose_bank_conflict_free")

    # --- Task 14 ---
    N = 1024
    in_arr = np.random.randn(N).astype(np.float32)
    num_blocks = math.ceil(N / 256)
    block_sums = np.zeros(num_blocks, dtype=np.float32)
    reduction_sum_shared[num_blocks, 256](in_arr, block_sums)
    np.testing.assert_allclose(np.sum(block_sums), np.sum(in_arr), rtol=1e-4)
    print("[PASSED] Task 14: reduction_sum_shared")

    # --- Task 15 ---
    N = 1024
    in_arr = np.random.randn(N).astype(np.float32)
    num_blocks = math.ceil(N / 256)
    block_max_vals = np.zeros(num_blocks, dtype=np.float32)
    block_max_indices = np.zeros(num_blocks, dtype=np.int64)
    reduction_argmax_shared[num_blocks, 256](in_arr, block_max_vals, block_max_indices)
    max_idx = np.argmax(block_max_vals)
    np.testing.assert_equal(block_max_vals[max_idx], np.max(in_arr))
    np.testing.assert_equal(block_max_indices[max_idx], np.argmax(in_arr))
    print("[PASSED] Task 15: reduction_argmax_shared")

    print("==================================================================")
    print("         ALL 15 TASKS PASSED AUTOMATED VERIFICATION!             ")
    print("==================================================================")

         RUNNING VERIFICATION FOR TASKS 1 TO 15                   
[PASSED] Task 01: vector_add_1d


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:1089: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 64 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:1089: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))


[PASSED] Task 02: vector_scale_grid_stride
[PASSED] Task 03: matrix_add_2d


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:1089: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:1089: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:1089: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))


[PASSED] Task 04: matrix_transpose_naive
[PASSED] Task 05: volume_init_3d


...

/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:1089: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 64 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


[PASSED] Task 11: conv_1d_shared_tiled


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:1089: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 32 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:1089: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


[PASSED] Task 12: matmul_tiled_shared
[PASSED] Task 13: transpose_bank_conflict_free
[PASSED] Task 14: reduction_sum_shared
[PASSED] Task 15: reduction_argmax_shared
         ALL 15 TASKS PASSED AUTOMATED VERIFICATION!             


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:1089: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:1089: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))
